In [1]:
print("Hello")

Hello


In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

text = "A simple, free, and efficient way to vectorize text."

vector = model.encode(f"search_query: {text}")

print(f"Object Type: {type(vector)}")  # Output: <class 'numpy.ndarray'>
print(f"Vector Dimensions: {vector.shape}")  # Output: (768,)
print(f"Sample values: {vector[:5]}")

<All keys matched successfully>
[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


Object Type: <class 'numpy.ndarray'>
Vector Dimensions: (768,)
Sample values: [ 1.3932841   1.0728705  -3.9158149  -0.9002627   0.01798691]


In [6]:
from functions import text_vectorize_score

result = text_vectorize_score("I have a cute little doll", "I wish my dog was gray.")
print(result["cosine_similarity"])
print(result["BERTScore_F1"].item())
print(result["BERTScore_P"].item())
print(result["BERTScore_R"].item())

<All keys matched successfully>
Loading weights: 100%|██████████| 100/100 [00:00<?, ?it/s]

0.48466048
0.7399899959564209
0.7310744524002075
0.7491257190704346


In [9]:
from functions import code_vectorize_score

code_result = code_vectorize_score("factorial = lambda n: 1 if n <= 1 else n * factorial(n - 1)", "def is_palindrome(s): return s == s[::-1]")
print(code_result["cosine_similarity"])
print(code_result["BERTScore_F1"])
print(code_result["BERTScore_P"])
print(code_result["BERTScore_R"])

Loading weights: 100%|██████████| 199/199 [00:00<?, ?it/s]

tensor([0.2558])
tensor([0.6242])
tensor([0.6398])
tensor([0.6093])


In [11]:
from functions import data_completion_score

data_result = data_completion_score(185000, 233200)
print(data_result)

2323240000


In [1]:
import os
from dotenv import load_dotenv
import anthropic
from functions import text_vectorize_score
from sentence_transformers import SentenceTransformer
from bert_score import BERTScorer
from tqdm.notebook import tqdm

model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
bert_scorer = BERTScorer(lang="en", model_type="distilbert-base-uncased")

load_dotenv(override=True)

api_key = os.getenv("ANTHROPIC_API_KEY")

if not api_key:
    raise ValueError("ANTHROPIC_API_KEY is missing from environment variables!")

client = anthropic.Anthropic(api_key=api_key)

message = """
    Analyze the story and finish the last sentence. Your response should only be 
    ONE sentence which is the final sentence that completes the story:

    Sherry hates basketball. 
    Sherry's boyfriend Tom loves basketball. 
    Sherry tries to learn more about basketball to make Tom happy. 
    For Tom's birthday she surprises him with tickets to a game. 
"""

ground_truth = "Sherry attends her first basketball game with Tom."

F1_array = []
cosine_array = []

for i in tqdm(range(30)):
    try:
        response = client.messages.create(
            model="claude-opus-4-8",
            max_tokens=500,
            output_config={
                "effort":"medium"
            },
            messages=[
                {"role": "user", "content": message}
            ]
        )
        #print(response.content[0].text)
        prediction = response.content[0].text

        result = text_vectorize_score(prediction, ground_truth, model, bert_scorer)

        F1_array.append(result["BERTScore_F1"])
        cosine_array.append(float(result["cosine_similarity"]))

    except anthropic.APIConnectionError as e:
        print("The server could not be reached", e.__cause__)
    except anthropic.RateLimitError:
        print("A 429 status code was received; consider backoff optimization.")
    except anthropic.APIStatusError as e:
        print(f"Another non-200 response was received (Status: {e.status_code})")
        print(e.response) 



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

<All keys matched successfully>


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

In [2]:
import numpy as np

print(cosine_array)
print(F1_array)

cos_mean_val = np.mean(cosine_array)
cos_median_val = np.median(cosine_array)
cos_q1 = np.percentile(cosine_array, 25)
cos_q3 = np.percentile(cosine_array, 75)
cos_iqr_val = cos_q3 - cos_q1
cos_data_range = np.ptp(cosine_array)
cos_std_dev = np.std(cosine_array)
cos_lower_whisker = cos_q1 - (1.5 * cos_iqr_val)
cos_upper_whisker = cos_q3 + (1.5 * cos_iqr_val)

F1_mean_val = np.mean(F1_array)
F1_median_val = np.median(F1_array)
F1_q1 = np.percentile(F1_array, 25)
F1_q3 = np.percentile(F1_array, 75)
F1_iqr_val = F1_q3 - F1_q1
F1_lower_whisker = F1_q1 - (1.5 * F1_iqr_val)
F1_upper_whisker = F1_q3 + (1.5 * F1_iqr_val)
F1_data_range = np.ptp(F1_array)
F1_std_dev = np.std(F1_array)

print(f"=======================================")
print(f"Cosine Similarity\n")
print(f"Mean (Average):     {cos_mean_val:.4f}")
print(f"Median (Middle):    {cos_median_val:.4f}")
print(f"25th Percentile(Q1): {cos_q1:.4f}")
print(f"75th Percentile(Q3): {cos_q3:.4f}")
print(f"IQR (Q3 - Q1):      {cos_iqr_val:.4f}")
print(f"Lower Whisker Limit: {cos_lower_whisker:.4f}")
print(f"Upper Whisker Limit: {cos_upper_whisker:.4f}")
print(f"Range (Max - Min):  {cos_data_range:.4f}")
print(f"Std Deviation:      {cos_std_dev:.4f}\n")

print(f"=======================================")
print(f"F1 Score\n")
print(f"Mean (Average):     {F1_mean_val:.4f}")
print(f"Median (Middle):    {F1_median_val:.4f}")
print(f"25th Percentile(Q1): {F1_q1:.4f}")
print(f"75th Percentile(Q3): {F1_q3:.4f}")
print(f"IQR (Q3 - Q1):      {F1_iqr_val:.4f}")
print(f"Lower Whisker Limit: {F1_lower_whisker:.4f}")
print(f"Upper Whisker Limit: {F1_upper_whisker:.4f}")
print(f"Range (Max - Min):  {F1_data_range:.4f}")
print(f"Std Deviation:      {F1_std_dev:.4f}")


[0.7090531587600708, 0.6736434698104858, 0.6668286919593811, 0.7209139466285706, 0.7575675845146179, 0.6257491111755371, 0.7335495352745056, 0.74120032787323, 0.74120032787323, 0.6851850152015686, 0.7187081575393677, 0.6944640874862671, 0.8237086534500122, 0.74120032787323, 0.74120032787323, 0.6939595341682434, 0.6736434698104858, 0.74120032787323, 0.6851850152015686, 0.7209139466285706, 0.8359820246696472, 0.7789390683174133, 0.6851850152015686, 0.7764236330986023, 0.6391317844390869, 0.6668286919593811, 0.74120032787323, 0.74120032787323, 0.6944640874862671, 0.770348846912384]
[0.7636021971702576, 0.7283916473388672, 0.7268772721290588, 0.7533935904502869, 0.7572005391120911, 0.7229713797569275, 0.7457626461982727, 0.7494066953659058, 0.7494066953659058, 0.735022783279419, 0.7503646016120911, 0.7601673007011414, 0.7580910325050354, 0.7494066953659058, 0.7494066953659058, 0.7317767143249512, 0.7283916473388672, 0.7494066953659058, 0.735022783279419, 0.7533935904502869, 0.7982062101364